# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a step-by-step template for loading and exploring the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided as a Croissant schema via the following URL:`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data LoadingLoad the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Accessing metadata as an object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\nPublished: {meta.date_published}")

## 2. Data OverviewReview the available record sets, their fields, and corresponding `@id` values. All references will be made by their `@id` as required by Croissant.

In [ ]:
# Discover available record sets and their fields via Croissant metadata

# We'll print each record set with its `@id` and its fields (and their `@id`s)
print('Record Sets:')
for record_set in dataset.metadata.find_all('@type', 'RecordSet'):
    print(f"  - RecordSet @id: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            # field can be a dict or a @id string
            if isinstance(field, dict):
                print(f"      Field @id: {field.get('@id')}  label: {field.get('rdfs:label', field.get('schema:name', ''))}")
            else:
                print(f"      Field @id: {field}")
    print()
# For exploration in the next step, collect the first record_set's @id
record_sets_ids = [record_set['@id'] for record_set in dataset.metadata.find_all('@type', 'RecordSet')]
if record_sets_ids:
    print(f"Discovered Record Set IDs: {record_sets_ids}")
else:
    print("No record sets discovered in this schema.")

## 3. Data ExtractionLoad the data from a chosen RecordSet into a Pandas DataFrame for further analysis. Use RecordSet and field `@id` values from the overview.

In [ ]:
# First, list all available record set @id values
record_sets = [rs['@id'] for rs in dataset.metadata.find_all('@type', 'RecordSet')]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")

# For continuing analysis, use the first record set as an example if available
if record_sets:
    chosen_record_set_id = record_sets[0]
    print(f"\nSample data from RecordSet @id: {chosen_record_set_id}")
    print(dataframes[chosen_record_set_id].head())
else:
    chosen_record_set_id = None

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps such as filtering, normalization, and grouping. Here, we'll demonstrate filtering on a numeric field, normalizing it, and grouping by another relevant field. All field names are referenced by their `@id` (column names as loaded).

In [ ]:
# Choose a numeric field for EDA (use column names from extraction step)

if chosen_record_set_id is not None and not dataframes[chosen_record_set_id].empty:
    df = dataframes[chosen_record_set_id]
    print('Available fields:', df.columns.tolist())

    # Heuristically pick a likely numeric field (e.g. 'cr:peptide_count' or 'cr:MW')
    # If not available, use the first float/integer-looking column
    possible_numeric_cols = [c for c in df.columns if (df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c]))]
    if not possible_numeric_cols:
        # Try to infer numeric fields (columns with only numbers)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        possible_numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    if possible_numeric_cols:
        numeric_field_id = possible_numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() != 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Choose a grouping field (non-numeric, e.g. 'cr:accession' or 'cr:description' or first non-numeric column)
        possible_group_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if possible_group_cols:
            group_field = possible_group_cols[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
            print(grouped_df.head())
        else:
            print("No grouping field found.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No data available for EDA.")

## 5. VisualizationVisualize data distributions and relationships. Modify the field names to use valid `@id`s/columns from the data above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of normalized numeric field
if chosen_record_set_id is not None and 'filtered_df' in locals() and not filtered_df.empty:
    field_for_plot = f"{numeric_field_id}_normalized"

    plt.figure(figsize=(8,6))
    sns.histplot(filtered_df[field_for_plot].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of normalized {numeric_field_id} (filtered)")
    plt.xlabel(f"{field_for_plot}")
    plt.ylabel("Count")
    plt.show()

    # Scatterplot of numeric_field_id vs group_field (if the group field is set)
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sample = filtered_df.head(50)  # for illustration, plot max 50
        sns.scatterplot(data=sample, x=group_field, y=numeric_field_id)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field_id} by {group_field} (first 50 records)")
        plt.tight_layout()
        plt.show()
else:
    print("No filtered data available to plot.")

## 6. ConclusionIn this notebook, we've loaded the FAIR\($^2$\) dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* via its Croissant schema, explored available RecordSets and fields (referencing them by their `@id`), extracted data into DataFrames, and performed basic exploratory data analysis and visualizations.Key findings and observations:
- The dataset contains annotated mass spectrometry protein data with rich metadata and multiple fields.
- Using the Croissant schema, dataset elements are referenced and loaded systematically by their `@id`.
- Numeric fields such as abundance, molecular weight, or peptide counts can be filtered and normalized for analysis.
- Grouping and visualization provide insights into data structure and highlight patterns for downstream research.**Next steps:** For advanced analyses, consult the Croissant schema for detailed field semantics and consider domain-specific data processing or integration with other proteomics resources.